# Building a Domain-Aligned Legal AI Assistant with Nugen

This cookbook demonstrates how to transform a general-purpose LLM into a
domain-aligned, benchmarked, production-ready model using Nugen APIs.


## End-to-End Flow

Document Upload → Benchmark Creation → Domain Alignment → Deployment → Inference


# Setup

In [6]:
!pip install requests

NUGEN INTELLIGENCE
Setup the nugen client using nugen base apis, api keys and headers

Refer to the docs.nugen.in for more information

In [ ]:
import requests
import time
import os
from dotenv import load_dotenv
load_dotenv()  # Load environment variables from .env file

BASE_URL = "https://api.nugen.in"
API_KEY = os.getenv("NUGEN_API_KEY")  # Get API key from environment variable

HEADERS = {
    "Authorization": f"Bearer {API_KEY}",
    "Content-Type": "application/json",
}


# Upload Document
To start with an alignment process lets upload a document on which we will align our model.

In [ ]:
upload_url = f"{BASE_URL}/api/v3/documents/"
files = {
    "files": (
        "file_name",
        open("sample.txt", "rb"),
        "text/plain",
    )
}

upload_headers = {"Authorization": f"Bearer {API_KEY}"}

response = requests.post(upload_url, headers=upload_headers, files=files)
print(response.text)

doc_id = response.json()["documents"][0]
print("Document ID:", doc_id)

# Create Benchmark

Once the document is uploaded successfully use that document to generate benchmark so that we can evaluate aligned model performance later

In [ ]:
benchmark_url = f"{BASE_URL}/api/v3/benchmark/create"

payload = {
    "documents": [doc_id],
    "num_questions": 10
}

response = requests.post(benchmark_url, headers=HEADERS, json=payload)
response.raise_for_status()

print(response.text)
benchmark_id = response.json()["id"]
print("Benchmark ID:", benchmark_id)

# Check Benchmark Status

Wait for the benchmark status to show Completed

# Poll Benchmark Status

In [ ]:
status_url = f"{BASE_URL}/api/v3/benchmark/status/{benchmark_id}"

response = requests.get(status_url, headers=HEADERS)

response.raise_for_status()

while True:
  status = requests.get(status_url, headers=HEADERS).json()
  print("benchmark:", status["status"])
  if status["status"] == "completed":
    status["id"] = benchmark_id
    break
  time.sleep(30)

print(benchmark_id)
print(response.text)


# Create Alignment

Once the benchmark is generated we will now align our model based on the same document(s) that we used earlier.

I am using Nugen-Flash-Instruct as the base model

In [ ]:
alignment_url = f"{BASE_URL}/api/v3/alignment-project/create"

payload = {
    "name": "test-domain-alignment-02",
    "baseModel": "nugen-flash-instruct",
    "benchmarkId": benchmark_id,
    "document_ids": [doc_id],
    "description": "Legal domain aligned model"
}

response = requests.post(alignment_url, headers=HEADERS, json=payload)
# response.raise_for_status()
alignment_id = response.json()["id"]
print(alignment_id)
print(response.text)

# Poll Alignment

Keep polling the alignment status endpoint until the status shows completed

In [ ]:
status_url = f"{BASE_URL}/api/v3/alignment-project/status/{alignment_id}"

response = requests.get(status_url, headers=HEADERS)
response.raise_for_status()
while True:
  status = requests.get(status_url, headers=HEADERS).json()
  print("Alignment:", status["status"])
  if status["status"] == "completed":
    model_id = status["data"]["model_id"]
    break
  time.sleep(30)

print(response.text)
print(model_id)

# Deploy Model

Now once the model is aligned use that aligned model to id to deploy the model so that we can perform an inference with it

In [ ]:
deploy_url = f"{BASE_URL}/api/v3/models/deploy-model/{model_id}"

requests.post(deploy_url, headers=HEADERS)

status_url = f"{BASE_URL}/api/v3/models/deploy-model/{model_id}/status"

while True:
    status = requests.get(status_url, headers=HEADERS).json()
    print("Deployment:", status["status"])
    if status["status"] == "completed":
        break
    time.sleep(10)


# Inference

Once the deployment is successful use that model id to perform inference.

In [ ]:
chat_url = f"{BASE_URL}/api/v3/inference/chat/completions"

payload = {
    "model": model_id,
    "messages": [
        {"role": "user", "content": "What is the purpose of the this file?"}
    ],
    "max_tokens": 300,
    "temperature": 0.2
}

response = requests.post(chat_url, headers=HEADERS, json=payload)
print(response.json())


# Troubleshooting

### Common Issues

• 401 Unauthorized → Check API key  
• 422 Validation Error → Payload mismatch  
• Model not responding → Ensure deployment status is `deployed`


# Key Takeaways

• Domain alignment reduces hallucinations  
• Benchmarks introduce measurable trust  
• Nugen APIs enable full automation  
• Suitable for regulated & enterprise AI